In [ ]:
!pip install monai torch

In [ ]:
import torch
from torch import nn
import torch.nn.functional as F
from monai.apps import DecathlonDataset

from monai.losses import DiceLoss
from monai.metrics import DiceMetric

from monai.transforms import (
    Activations,
    Activationsd,
    AsDiscrete,
    AsDiscreted,
    Compose,
    Invertd,
    LoadImaged,
    MapTransform,
    NormalizeIntensityd,
    Orientationd,
    RandFlipd,
    RandScaleIntensityd,
    RandShiftIntensityd,
    RandSpatialCropd,
    Spacingd,
    EnsureTyped,
    EnsureChannelFirstd,
)
import time
from monai.data import DataLoader, decollate_batch
from monai.inferers import sliding_window_inference

<frozen importlib._bootstrap_external>:1301: FutureWarning: The cuda.cudart module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.runtime module instead.


In [ ]:
class DoubleConv(nn.Module):
    """(convolution => [BN] => ReLU) * 2"""
    def __init__(self, in_channels, out_channels, mid_channels=None):
        super().__init__()
        if not mid_channels:
            mid_channels = out_channels
        self.double_conv = nn.Sequential(
            nn.Conv3d(in_channels, mid_channels, kernel_size=3, padding=1),
            nn.BatchNorm3d(mid_channels),
            nn.LeakyReLU(inplace=True),
            nn.Conv3d(mid_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm3d(out_channels),
            nn.LeakyReLU(inplace=True),
        )

    def forward(self, x):
        return self.double_conv(x)

In [ ]:
class Up(nn.Module):
    """Upscaling then double conv"""
    def __init__(self, in_channels_from_down_path, skip_channels, out_channels, bilinear=True):
        super().__init__()

        # if bilinear, use the normal convolutions to reduce the number of channels
        if bilinear:
            self.up = nn.Upsample(scale_factor=2,
                                  mode='trilinear',
                                  align_corners=True)
            # The DoubleConv should take in_channels_from_down_path (x1) + skip_channels (x2) as its input
            # After upsampling, x1 still has in_channels_from_down_path
            self.conv = DoubleConv(in_channels_from_down_path + skip_channels, out_channels, (in_channels_from_down_path + skip_channels) // 2)
        else:
            # If not bilinear, transposed conv halves channels of x1.
            self.up = nn.ConvTranspose3d(
                in_channels_from_down_path,
                in_channels_from_down_path // 2,
                kernel_size=2,
                stride=2,
            )
            # Then concatenate with skip_channels.
            self.conv = DoubleConv(in_channels_from_down_path // 2 + skip_channels, out_channels)

    def forward(self, x1, x2):
        x1 = self.up(x1)
        # input is CHWD
        diffY = x2.size()[2] - x1.size()[2]
        diffX = x2.size()[3] - x1.size()[3]
        diffZ = x2.size()[4] - x1.size()[4]

        x1 = F.pad(
            x1,
            [diffX // 2, diffX - diffX // 2,
             diffY // 2, diffY - diffY // 2,
             diffZ // 2, diffZ - diffZ // 2])

        x = torch.cat([x2, x1], dim=1)
        return self.conv(x)

In [ ]:

class Down(nn.Module):
    """Downscaling with maxpool then double conv"""
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.maxpool_conv = nn.Sequential(
            nn.MaxPool3d(2),
            DoubleConv(in_channels, out_channels),
        )

    def forward(self, x):
        return self.maxpool_conv(x)

In [ ]:
from monai.networks.blocks import PatchEmbeddingBlock
class MultiHeadSelfAttention(nn.Module):
    def __init__(self, channel, spatial_dims, patch_size, hidden):
        super(MultiHeadSelfAttention, self).__init__()
        #self.query = MultiHeadDense(channel, bias=False)
        #self.key = MultiHeadDense(channel, bias=False)
        #self.value = MultiHeadDense(channel, bias=False)
        #self.softmax = nn.Softmax(dim=1)
        self.pe = PatchEmbeddingBlock(channel, spatial_dims, patch_size, hidden, 1)
        self.attn = torch.nn.MultiheadAttention(channel, 1, batch_first=True)

    def forward(self, x):
        b, c, h, w, d = x.size()
        # pe = self.positional_encoding_2d(c, h, w)
        x = self.pe(x)
        #x = x + pe
        #x = x.reshape(b, c, h * w).permute(0, 2, 1)  #[b, h*w, d]
        #Q = self.query(x)
        #K = self.key(x)
        #A = self.softmax(torch.bmm(Q, K.permute(0, 2, 1)) /
        #                 math.sqrt(c))  #[b, h*w, h*w]
        #V = self.value(x)
        #x = torch.bmm(A, V).permute(0, 2, 1).reshape(b, c, h, w)
        #print(x.size())
        attn_output = self.attn(x, x, x, need_weights=False)[0]
        x = attn_output.transpose(-1, -2).reshape(b, c, h, w, d)
        return x

In [ ]:
class TransformerUp(nn.Module):
    def __init__(self, Ychannels, Schannels, spat_dimS, spat_dimY, num_heads):
        super(TransformerUp, self).__init__()
        self.MHCA = MultiHeadCrossAttention(Ychannels, Schannels, spat_dimS, spat_dimY, num_heads)
        self.conv = nn.Sequential(
            nn.Conv3d(Ychannels,
                      Schannels,
                      kernel_size=3,
                      stride=1,
                      padding=1,
                      bias=True), nn.BatchNorm3d(Schannels),
            nn.LeakyReLU(inplace=True),
            nn.Conv3d(Schannels,
                      Schannels,
                      kernel_size=3,
                      stride=1,
                      padding=1,
                      bias=True), nn.BatchNorm3d(Schannels),
            nn.LeakyReLU(inplace=True))

    def forward(self, Y, S):
        x = self.MHCA(Y, S)
        x = self.conv(x)
        return x

In [ ]:
class OutConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(OutConv, self).__init__()
        self.conv = nn.Conv3d(in_channels, out_channels, kernel_size=1)

    def forward(self, x):
        return self.conv(x)

In [ ]:
class Encoder(nn.Module):
    def __init__(self, in_channels):
        super(Encoder, self).__init__()

        self.inc = DoubleConv(in_channels, 64)
        self.down1 = Down(64, 128)
        self.down2 = Down(128, 256)
        self.down3 = Down(256, 512)

    def forward(self, x):
        #print(x.size())
        x1 = self.inc(x)
        #print(x1.size())
        x2 = self.down1(x1)
        #print(x2.size())
        x3 = self.down2(x2)
        #print(x3.size())
        x4 = self.down3(x3)

        return x1, x2, x3, x4


In [ ]:
class Encoder_VAE(nn.Module):
    def __init__(self, in_channels):
        super(Encoder_VAE, self).__init__()

        self.inc = DoubleConv(in_channels, 64)
        self.down1 = Down(64, 128)
        self.down2 = Down(128, 256)
        self.down3 = Down(256, 512)

    def forward(self, x):
        #print(x.size())
        x1 = self.inc(x)
        #print(x1.size())
        x2 = self.down1(x1)
        #print(x2.size())
        x3 = self.down2(x2)
        #print(x3.size())
        x4 = self.down3(x3)

        return x1, x2, x3, x4

In [ ]:
class GaussianSampler(nn.Module):
    """
    This predicts the mean and logvariance parameters,
    then generates an approximate sample from the posterior.
    Converted from TensorFlow to PyTorch.
    """

    def __init__(self, name='gaussian_sampler'):
        super(GaussianSampler, self).__init__()

    def forward(self, means, logvars, list_mod, choices, is_inference):
        """
        Args:
            means: dict with modality keys -> mean tensors
            logvars: dict with modality keys -> log variance tensors
            list_mod: list of modality names to use
            choices: boolean tensor or list indicating which modalities are available
            is_inference: bool, if True returns only means without sampling

        Returns:
            sampled latent vector
        """
        eps = 1e-7
        device = means[list_mod[0]].device

        # Get prior parameters
        mu_prior = torch.zeros_like(means[list_mod[0]])
        log_prior = torch.zeros_like(means[list_mod[0]])

        # Convert choices to tensor if needed
        if not isinstance(choices, torch.Tensor):
            choices = torch.tensor(choices, device=device, dtype=torch.bool)

        # Compute precision-weighted means and precisions
        # T is the sum of precisions: 1/variance
        T_list = []
        mu_list = []

        for mod in list_mod:
            precision = 1.0 / (torch.exp(logvars[mod]) + eps)
            precision_weighted_mean = means[mod] / (torch.exp(logvars[mod]) + eps)

            T_list.append(precision)
            mu_list.append(precision_weighted_mean)

        # Apply choices mask (select available modalities)
        T = torch.stack(T_list, dim=0)
        mu = torch.stack(mu_list, dim=0)

        # Convert choices to appropriate shape for masking
        choices_expanded = choices.view(-1, 1, 1, 1) if len(T.shape) > 1 else choices.view(-1, 1)

        T = T[choices]
        mu = mu[choices]

        # Add prior precision and prior mean
        T = torch.cat([T, 1.0 + log_prior.unsqueeze(0)], dim=0)
        mu = torch.cat([mu, mu_prior.unsqueeze(0)], dim=0)

        # Compute posterior mean and variance
        T_sum = torch.sum(T, dim=0)
        posterior_means = torch.sum(mu, dim=0) / T_sum
        var = 1.0 / T_sum
        posterior_logvars = torch.log(var + eps)

        if is_inference:
            return posterior_means
        else:
            # Sample from posterior using reparameterization trick
            noise_sample = torch.randn_like(posterior_means)
            output = posterior_means + torch.exp(0.5 * posterior_logvars) * noise_sample
            return output

In [ ]:
class MultiHeadCrossAttention(nn.Module):
    def __init__(self, channelY, spat_dimY, num_heads):
        super(MultiHeadCrossAttention, self).__init__()
        #self.Sconv = nn.Sequential(
        #    nn.MaxPool3d(2), nn.Conv3d(channelS, channelS, kernel_size=1),
        #    nn.BatchNorm3d(channelS), nn.LeakyReLU(inplace=True))
        #self.Yconv = nn.Sequential(
        #    nn.Conv3d(channelY, channelS, kernel_size=1),
        #    nn.BatchNorm3d(channelS), nn.LeakyReLU(inplace=True))
        #self.query = MultiHeadDense(channelS, bias=False)
        #self.key = MultiHeadDense(channelS, bias=False)
        #self.value = MultiHeadDense(channelS, bias=False)
        #self.conv = nn.Sequential(
        #    nn.Conv3d(channelY, channelS, kernel_size=1),
        #    nn.BatchNorm3d(channelS), nn.ReLU(inplace=True),
        #    nn.Upsample(scale_factor=4, mode='trilinear', align_corners=True))
        #self.Yconv2 = nn.Sequential(
        #    nn.Upsample(scale_factor=2, mode='trilinear', align_corners=True),
        #    nn.Conv3d(channelY, channelY, kernel_size=3, padding=1),
        #    nn.Conv3d(channelY, channelS, kernel_size=1),
        #    nn.BatchNorm3d(channelS), nn.LeakyReLU(inplace=True))
        self.upsample = nn.Upsample(scale_factor=2, mode='trilinear', align_corners=True)

        self.embedding1 = PatchEmbeddingBlock(channelY, spat_dimY, (2, 2, 2), channelY, num_heads)
        self.embedding2 = PatchEmbeddingBlock(channelY, spat_dimY, (2, 2, 2), channelY, num_heads)
        self.embedding3 = PatchEmbeddingBlock(channelY, spat_dimY, (2, 2, 2), channelY, num_heads)
        self.embedding4 = PatchEmbeddingBlock(channelY, spat_dimY, (2, 2, 2), channelY, num_heads)

        #self.Ype = PatchEmbeddingBlock(channelY, spat_dimY, (2, 2, 2), channelY, num_heads)

        self.attn1 = torch.nn.MultiheadAttention(channelY, num_heads, batch_first=True)
        self.attn2 = torch.nn.MultiheadAttention(channelY, num_heads, batch_first=True)
        self.attn3 = torch.nn.MultiheadAttention(channelY, num_heads, batch_first=True)
        self.reshape = lambda x: torch.reshape(x.permute(0, 2, 1),( 1, channelY, spat_dimY[0] // 2,
                                               spat_dimY[1] // 2, spat_dimY[2] // 2))
        #self.attn1 = torch.nn.MultiheadAttention(channelY, num_heads, batch_first=True)
        #self.softmax = nn.Softmax(dim=1)
        #self.Spe = PositionalEncodingPermute2D(channelS)
        #self.Ype = PositionalEncodingPermute2D(channelY)

    def forward(self, x1, x2, x3, x4):

        x1 = self.embedding1(x1)

        x2 = self.embedding2(x2)

        x3 = self.embedding3(x3)

        x4 = self.embedding4(x4)

        attn_output = self.attn1(x2, x1, x1, need_weights=False)[0]
        attn_output = self.attn2(x3, attn_output, attn_output, need_weights=False)[0]
        attn_output = self.attn3(x4, attn_output, attn_output, need_weights=False)[0]
        attn_3d = self.reshape(attn_output)
        #print("attention output: ", attn_output.size())

        return self.upsample(attn_3d)

In [ ]:
class U_Transformer(nn.Module):
    def __init__(self, in_channels, classes, bilinear=True):
        super(U_Transformer, self).__init__()
        self.in_channels = in_channels
        self.classes = classes
        self.bilinear = bilinear

        #self.inc = DoubleConv(in_channels, 64)
        #self.down1 = Down(64, 128)
        #self.down2 = Down(128, 256)
        #self.down3 = Down(256, 512)
        self.encoders = nn.ModuleList([Encoder(in_channels) for _ in range(4)])
        #self.transformer1 = MultiHeadSelfAttention(512, (28, 28, 18), (1, 1, 1), 512 )
        #self.MHSA = MultiHeadSelfAttention(512, )
        # Reverted spatial dimensions to match original roi_size=[224, 224, 144]
        self.cross_attn1 = MultiHeadCrossAttention(64, (224, 224, 144), 1)
        self.cross_attn2 = MultiHeadCrossAttention(128, (112, 112, 72), 1)
        self.cross_attn3 = MultiHeadCrossAttention(256, (56, 56, 36), 1)
        self.cross_attn4 = MultiHeadCrossAttention(512, (28, 28, 18), 1)
        #self.up1 = TransformerUp(512, 256, (56, 56, 36), (28, 28, 18), 1)
        #self.up2 = TransformerUp(256, 128, (112, 112, 72), (56, 56, 36), 1)
        #self.up3 = TransformerUp(128, 64, (224, 224, 144), (112, 112, 72), 1)
        self.outc = OutConv(64, classes)

        # Updated Up constructor calls with explicit skip_channels
        self.up1 = Up(in_channels_from_down_path=512, skip_channels=256, out_channels=256)
        self.up2 = Up(in_channels_from_down_path=256, skip_channels=128, out_channels=128)
        self.up3 = Up(in_channels_from_down_path=128, skip_channels=64, out_channels=64)
        #self.up1 = Up(512, 256)

    def forward(self, x):

        # Reverted reshape to match original roi_size=[224, 224, 144]
        #print(x.size())
        encodings = [self.encoders[i](x[:, i].reshape(1, 1, 224, 224, 144)) for i in range(4)]
        # Use the first modality for now
        fusion1 = self.cross_attn1(encodings[0][0], encodings[1][0], encodings[2][0], encodings[3][0])
        fusion2 = self.cross_attn2(encodings[0][1], encodings[1][1], encodings[2][1], encodings[3][1])
        fusion3 = self.cross_attn3(encodings[0][2], encodings[1][2], encodings[2][2], encodings[3][2])

        fusion4 = self.cross_attn4(encodings[0][3], encodings[1][3], encodings[2][3], encodings[3][3])
        #x4 = self.transformer1(x4)
        #print(fusion4.size(), fusion3.size())
        x = self.up1(fusion4, fusion3)
        x = self.up2(x, fusion2)
        x = self.up3(x, fusion1)
        logits = self.outc(x)

        return logits

In [ ]:
class U_Transformer_VAE(nn.Module):
    def __init__(self, in_channels, classes, bilinear=True):
        super(U_Transformer_VAE, self).__init__()
        self.in_channels = in_channels
        self.classes = classes
        self.bilinear = bilinear

        self.encoders = nn.ModuleList([Encoder_VAE(in_channels) for _ in range(4)])
        self.gaussians = nn.ModuleList([GaussianSampler() for _ in range(4)])

        # These cross-attention modules might become redundant if Gaussian sampling fuses modalities per level
        # However, they are kept for now as per the existing structure.
        #self.cross_attn1 = MultiHeadCrossAttention(64, (224, 224, 144), 1) # Note: this spatial dim is for level 0 (x1 output size from original ROI)
        #self.cross_attn2 = MultiHeadCrossAttention(128, (112, 112, 72), 1)
        #self.cross_attn3 = MultiHeadCrossAttention(256, (56, 56, 36), 1)
        self.cross_attn4 = MultiHeadCrossAttention(512, (28, 28, 18), 1)

        self.outc = OutConv(64, classes)

        self.up1 = Up(in_channels_from_down_path=512, skip_channels=128, out_channels=256)
        self.up2 = Up(in_channels_from_down_path=256, skip_channels=64, out_channels=128)
        self.up3 = Up(in_channels_from_down_path=128, skip_channels=32, out_channels=64)

    def forward(self, x):

        print("encoding")
        # all_encodings will be a list of 4 tuples, where each tuple contains (x1, x2, x3, x4) for one modality
        all_encodings = [self.encoders[i](x[:, i].reshape(1, 1, 224, 224, 144)) for i in range(4)]

        # Define the total channels for each encoder level output
        channels_per_level = [
            64,  # x1 (inc output)
            128, # x2 (down1 output)
            256, # x3 (down2 output)
            512  # x4 (down3 output)
        ]

        fused_encodings_per_level = []

        print("gaussian sampling per level")
        for level_idx in range(3): # Iterate through each encoding level (x1, x2, x3, x4)
            level_means = []
            level_logvars = []
            current_total_channels = channels_per_level[level_idx]
            half_channels = current_total_channels // 2 # Assuming first half for means, second for logvars

            for mod_idx in range(4): # Iterate through each modality
                # Get the specific encoding output for the current modality and level
                encoding_output_for_mod_level = all_encodings[mod_idx][level_idx]

                # Split into means and logvars. Ensure the output of Encoder_VAE has enough channels.
                # The Encoder_VAE's DoubleConv outputs 64, 128, 256, 512 channels, respectively.
                # So, half_channels will be 32, 64, 128, 256.
                level_means.append(encoding_output_for_mod_level[:, :half_channels, ...].unsqueeze(0))
                logvar_tensor = encoding_output_for_mod_level[:, half_channels:, ...]
                level_logvars.append(torch.log(logvar_tensor + 1e-7).unsqueeze(0))

            # Apply GaussianSampler for the current level, fusing across modalities
            fused_latent_for_level = self.gaussians[level_idx](
                torch.cat(level_means, dim=0),
                torch.cat(level_logvars, dim=0),
                torch.arange(0, 4, step=1),
                torch.tensor([True, True, True, True], dtype=torch.bool),
                not self.training
            )
            fused_encodings_per_level.append(fused_latent_for_level)

        # Unpack the fused latents for each level
        fused_x1, fused_x2, fused_x3 = fused_encodings_per_level

        # Now, apply cross-attention to the fused latents from Gaussian sampling
        # Note: Since fused_xN is already a single tensor, passing it 4 times to MultiHeadCrossAttention
        # will effectively make it a self-attention operation on the fused latent features.
        print("applying cross attention to fused latents")
        final_fusion4 = self.cross_attn4(all_encodings[0][3], all_encodings[0][3],
                                         all_encodings[0][3], all_encodings[0][3])

        # Adjusting the Up calls to directly use the final_fusionX after cross-attention
        x = self.up1(final_fusion4, fused_x3)
        x = self.up2(x, fused_x2)
        x = self.up3(x, fused_x1)
        logits = self.outc(x)

        return logits

In [ ]:
import os
os.environ['MONAI_DATA_DIRECTORY'] = 'monai_data'
directory = os.environ.get("MONAI_DATA_DIRECTORY")
if directory is not None:
    os.makedirs(directory, exist_ok=True)
root_dir = tempfile.mkdtemp() if directory is None else directory
print(root_dir)

monai_data


In [ ]:
class ConvertToMultiChannelBasedOnBratsClassesd(MapTransform):
    """
    Convert labels to multi channels based on brats classes:
    label 1 is the peritumoral edema
    label 2 is the GD-enhancing tumor
    label 3 is the necrotic and non-enhancing tumor core
    The possible classes are TC (Tumor core), WT (Whole tumor)
    and ET (Enhancing tumor).

    """

    def __call__(self, data):
        d = dict(data)
        for key in self.keys:
            result = []
            # merge label 2 and label 3 to construct TC
            result.append(torch.logical_or(d[key] == 2, d[key] == 3))
            # merge labels 1, 2 and 3 to construct WT
            result.append(torch.logical_or(torch.logical_or(d[key] == 2, d[key] == 3), d[key] == 1))
            # label 2 is ET
            result.append(d[key] == 2)
            d[key] = torch.stack(result, axis=0).float()
        return d

In [ ]:
train_transform = Compose(
    [
        # load 4 Nifti images and stack them together
        LoadImaged(keys=["image", "label"]),
        EnsureChannelFirstd(keys="image"),
        EnsureTyped(keys=["image", "label"]),
        ConvertToMultiChannelBasedOnBratsClassesd(keys="label"),
        Orientationd(keys=["image", "label"], axcodes="RAS"),
        Spacingd(
            keys=["image", "label"],
            pixdim=(1.0, 1.0, 1.0),
            mode=("bilinear", "nearest"),
        ),
        RandSpatialCropd(keys=["image", "label"], roi_size=[224, 224, 144], random_size=False),
        RandFlipd(keys=["image", "label"], prob=0.5, spatial_axis=0),
        RandFlipd(keys=["image", "label"], prob=0.5, spatial_axis=1),
        RandFlipd(keys=["image", "label"], prob=0.5, spatial_axis=2),
        NormalizeIntensityd(keys="image", nonzero=True, channel_wise=True),
        RandScaleIntensityd(keys="image", factors=0.1, prob=1.0),
        RandShiftIntensityd(keys="image", offsets=0.1, prob=1.0),
    ]
)

/usr/local/lib/python3.12/dist-packages/monai/utils/deprecate_utils.py:321: FutureWarning: monai.transforms.spatial.dictionary Orientationd.__init__:labels: Current default value of argument `labels=(('L', 'R'), ('P', 'A'), ('I', 'S'))` was changed in version None from `labels=(('L', 'R'), ('P', 'A'), ('I', 'S'))` to `labels=None`. Default value changed to None meaning that the transform now uses the 'space' of a meta-tensor, if applicable, to determine appropriate axis labels.
  warn_deprecated(argname, msg, warning_category)


In [ ]:
train_ds = DecathlonDataset(
    root_dir=root_dir,
    task="Task01_BrainTumour",
    transform=train_transform,
    section="training",
    download=True,
    cache_rate=0.0,
    num_workers=0,
)

2026-04-11 11:12:12,073 - INFO - Verified 'Task01_BrainTumour.tar', md5: 240a19d752f0d9e9101544901065d872.
2026-04-11 11:12:12,073 - INFO - File exists: monai_data/Task01_BrainTumour.tar, skipped downloading.
2026-04-11 11:12:12,074 - INFO - Non-empty folder exists in monai_data/Task01_BrainTumour, skipped extracting.


In [ ]:
os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Ensure previous model and tensors are explicitly deleted to free GPU memory
if 'model' in locals() and model is not None:
    del model
torch.cuda.empty_cache()

model = U_Transformer_VAE(1, 3).to(device)
batch_data = train_ds[0]
scaler = torch.cuda.amp.GradScaler()  # Re-enabled for mixed precision
# enable cuDNN benchmark
torch.backends.cudnn.benchmark = True
inputs, labels = (
            batch_data["image"].to(device),
            batch_data["label"].to(device),
        )
# Removed hardcoded reshape; model expects inputs for each modality to be 1 channel, handled internally
# The U_Transformer's forward method expects an input of shape (batch, modalities, H, W, D)
# Then it internally reshapes for each encoder.
# The DataLoader already provides batches with the correct shapes after transformations.
# For batch_size=1, inputs will be (1, 4, 224, 224, 144) and labels will be (1, 3, 224, 224, 144).
print(inputs.size())
print(labels.size())

inputs = inputs.resize(1, 4, 224, 224, 144)
labels = labels.resize(1, 3, 224, 224, 144)

loss_function = DiceLoss(smooth_nr=0, smooth_dr=1e-5, squared_pred=True, to_onehot_y=False, sigmoid=True)
optimizer = torch.optim.Adam(model.parameters(), 1e-4, weight_decay=1e-5)
#lr_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max_epochs)

dice_metric = DiceMetric(include_background=True, reduction="mean")
dice_metric_batch = DiceMetric(include_background=True, reduction="mean_batch")


model.train()
epoch_loss = 0
step = 0
step_start = time.time()
with torch.profiler.profile(
            activities=[
                torch.profiler.ProfilerActivity.CPU,
                torch.profiler.ProfilerActivity.CUDA,
            ],
            #schedule=torch.profiler.schedule(wait=1, warmup=1, active=3),
            on_trace_ready=torch.profiler.tensorboard_trace_handler('./log/profile'),
            with_stack=True
        ) as prof:
  with torch.cuda.amp.autocast():  # Re-enabled AMP
      outputs = model(inputs)
      loss = loss_function(outputs, labels)

  scaler.scale(loss).backward()  # Re-enabled scaler
  scaler.step(optimizer)  # Re-enabled scaler
  scaler.update()  # Re-enabled scaler
  #loss.backward()
  #optimizer.step()

  epoch_loss += loss.item()
  print(
      f"{step}/{len(train_ds)}"
      f", train_loss: {loss.item():.4f}"
      f", step time: {(time.time() - step_start):.4f}"
  )
              #prof.step()

  #print('outputs', outputs.shape)

/tmp/ipykernel_24601/2757669283.py:10: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()  # Re-enabled for mixed precision
/usr/local/lib/python3.12/dist-packages/torch/_tensor.py:1703: UserWarning: non-inplace resize is deprecated
  ret = func(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/profiler/profiler.py:217: UserWarning: Warning: Profiler clears events at the end of each cycle.Only events from the current cycle will be reported.To keep events across cycles, set acc_events=True.
  _warn_once(
/tmp/ipykernel_24601/2757669283.py:49: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():  # Re-enabled AMP


torch.Size([4, 224, 224, 144])
torch.Size([3, 224, 224, 144])
encoding
gaussian sampling per level
applying cross attention to fused latents
0/388, train_loss: nan, step time: 9.2646


In [ ]:
loss

metatensor(nan, device='cuda:0', grad_fn=<AliasBackward0>)

In [ ]:
class ConvertToMultiChannelBasedOnBratsClassesd(MapTransform):
    """
    Convert labels to multi channels based on brats classes:
    label 1 is the peritumoral edema
    label 2 is the GD-enhancing tumor
    label 3 is the necrotic and non-enhancing tumor core
    The possible classes are TC (Tumor core), WT (Whole tumor)
    and ET (Enhancing tumor).

    """

    def __call__(self, data):
        d = dict(data)
        for key in self.keys:
            result = []
            # merge label 2 and label 3 to construct TC
            result.append(torch.logical_or(d[key] == 2, d[key] == 3))
            # merge labels 1, 2 and 3 to construct WT
            result.append(torch.logical_or(torch.logical_or(d[key] == 2, d[key] == 3), d[key] == 1))
            # label 2 is ET
            result.append(d[key] == 2)
            d[key] = torch.stack(result, axis=0).float()
        return d

In [ ]:
train_transform = Compose(
    [
        # load 4 Nifti images and stack them together
        LoadImaged(keys=["image", "label"]),
        EnsureChannelFirstd(keys="image"),
        EnsureTyped(keys=["image", "label"]),
        ConvertToMultiChannelBasedOnBratsClassesd(keys="label"),
        Orientationd(keys=["image", "label"], axcodes="RAS"),
        Spacingd(
            keys=["image", "label"],
            pixdim=(1.0, 1.0, 1.0),
            mode=("bilinear", "nearest"),
        ),
        RandSpatialCropd(keys=["image", "label"], roi_size=[224, 224, 144], random_size=False), # Reverted ROI size
        RandFlipd(keys=["image", "label"], prob=0.5, spatial_axis=0),
        RandFlipd(keys=["image", "label"], prob=0.5, spatial_axis=1),
        RandFlipd(keys=["image", "label"], prob=0.5, spatial_axis=2),
        NormalizeIntensityd(keys="image", nonzero=True, channel_wise=True),
        RandScaleIntensityd(keys="image", factors=0.1, prob=1.0),
        RandShiftIntensityd(keys="image", offsets=0.1, prob=1.0),
    ]
)

val_transform = Compose(
    [
        LoadImaged(keys=["image", "label"]),
        EnsureChannelFirstd(keys="image"),
        EnsureTyped(keys=["image", "label"]),
        ConvertToMultiChannelBasedOnBratsClassesd(keys="label"),
        Orientationd(keys=["image", "label"], axcodes="RAS"),
        Spacingd(
            keys=["image", "label"],
            pixdim=(1.0, 1.0, 1.0),
            mode=("bilinear", "nearest"),
        ),
        # Add RandSpatialCropd for validation to ensure consistent input size
        RandSpatialCropd(keys=["image", "label"], roi_size=[224, 224, 144], random_size=False), # Reverted ROI size
        NormalizeIntensityd(keys="image", nonzero=True, channel_wise=True),
    ]
)

In [ ]:
train_ds = DecathlonDataset(
    root_dir=root_dir,
    task="Task01_BrainTumour",
    transform=train_transform,
    section="training",
    download=False,
    cache_rate=0.0,
    num_workers=0,
)
train_loader = DataLoader(train_ds, batch_size=1, shuffle=True, num_workers=0)
val_ds = DecathlonDataset(
    root_dir=root_dir,
    task="Task01_BrainTumour",
    transform=val_transform,
    section="validation",
    download=False,
    cache_rate=0.0,
    num_workers=0,
)
val_loader = DataLoader(val_ds, batch_size=1, shuffle=False, num_workers=0)

In [ ]:
max_epochs = 300
val_interval = 1
VAL_AMP = True

device = torch.device("cuda:0")
model = U_Transformer(1, 3).to(device)
loss_function = DiceLoss(smooth_nr=0, smooth_dr=1e-5, squared_pred=True, to_onehot_y=False, sigmoid=True)
optimizer = torch.optim.Adam(model.parameters(), 1e-4, weight_decay=1e-5)
lr_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max_epochs)

dice_metric = DiceMetric(include_background=True, reduction="mean")
dice_metric_batch = DiceMetric(include_background=True, reduction="mean_batch")

post_trans = Compose([Activations(sigmoid=True), AsDiscrete(threshold=0.5)])


# define inference method
def inference(input):
    def _compute(input):
        return sliding_window_inference(
            inputs=input,
            roi_size=(224, 224, 144), # Reverted to match original training crop size
            sw_batch_size=1,
            predictor=model,
            overlap=0.5,
        )

    if VAL_AMP:
        with torch.autocast("cuda"):
            return _compute(input)
    else:
        return _compute(input)


# use amp to accelerate training
scaler = torch.GradScaler("cuda")
# enable cuDNN benchmark
torch.backends.cudnn.benchmark = True

In [ ]:
best_metric = -1
best_metric_epoch = -1
best_metrics_epochs_and_time = [[], [], []]
epoch_loss_values = []
metric_values = []
metric_values_tc = []
metric_values_wt = []
metric_values_et = []
#set-up model
device = torch.device("cuda:0")
#model = U_Transformer(1, 3).to(device)
torch.cuda.empty_cache()

# Re-initialize optimizer, loss function, metrics, and scaler after model setup
loss_function = DiceLoss(smooth_nr=0, smooth_dr=1e-5, squared_pred=True, to_onehot_y=False, sigmoid=True)
optimizer = torch.optim.Adam(model.parameters(), 1e-4, weight_decay=1e-5)
# lr_scheduler is also defined in 1SiHZ1k9PEwo, re-initialize for consistency if needed, but not directly causing this error.
lr_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max_epochs)
dice_metric = DiceMetric(include_background=True, reduction="mean")
dice_metric_batch = DiceMetric(include_background=True, reduction="mean_batch")
scaler = torch.GradScaler("cuda") # Re-initialize the scaler

total_start = time.time()
accum_iter = 10
for epoch in range(max_epochs):
    epoch_start = time.time()
    print("-" * 10)
    print(f"epoch {epoch + 1}/{max_epochs}")
    model.train()
    epoch_loss = 0
    step = 0
    for batch_data in train_loader:
        step_start = time.time()

        inputs, labels = (
            batch_data["image"].to(device),
            batch_data["label"].to(device),
        )

        """with torch.profiler.profile(
            activities=[
                torch.profiler.ProfilerActivity.CPU,
                torch.profiler.ProfilerActivity.CUDA,
            ],
            schedule=torch.profiler.schedule(wait=1, warmup=1, active=3),
            on_trace_ready=torch.profiler.tensorboard_trace_handler('./log/profile'),
            with_stack=True
        ) as prof:"""
        with torch.autocast("cuda"):
            outputs = model(inputs)
            loss = loss_function(outputs, labels) / accum_iter
        scaler.scale(loss).backward()

        if ((step + 1) % accum_iter == 0) or (step + 1 == len(train_loader)):
          scaler.step(optimizer)
          scaler.update()
          optimizer.zero_grad()
          epoch_loss += loss.item() * accum_iter

          print(
              f"{step}/{len(train_ds) // train_loader.batch_size}"
              f", train_loss: {(loss.item() * accum_iter):.4f}"
              f", step time: {(time.time() - step_start):.4f}"
          )

        step += 1
            #prof.step()

    lr_scheduler.step()
    epoch_loss /= step
    epoch_loss_values.append(epoch_loss)
    print(f"epoch {epoch + 1} average loss: {epoch_loss:.4f}")

    if (epoch + 1) % val_interval == 0:
        model.eval()
        with torch.no_grad():
            for val_data in val_loader:
                val_inputs, val_labels = (
                    val_data["image"].to(device),
                    val_data["label"].to(device),
                )
                val_outputs = inference(val_inputs)
                val_outputs = [post_trans(i) for i in decollate_batch(val_outputs)]
                dice_metric(y_pred=val_outputs, y=val_labels)
                dice_metric_batch(y_pred=val_outputs, y=val_labels)

            metric = dice_metric.aggregate().item()
            metric_values.append(metric)
            metric_batch = dice_metric_batch.aggregate()
            metric_tc = metric_batch[0].item()
            metric_values_tc.append(metric_tc)
            metric_wt = metric_batch[1].item()
            metric_values_wt.append(metric_wt)
            metric_et = metric_batch[2].item()
            metric_values_et.append(metric_et)
            dice_metric.reset()
            dice_metric_batch.reset()

            if metric > best_metric:
                best_metric = metric
                best_metric_epoch = epoch + 1
                best_metrics_epochs_and_time[0].append(best_metric)
                best_metrics_epochs_and_time[1].append(best_metric_epoch)
                best_metrics_epochs_and_time[2].append(time.time() - total_start)
                torch.save(
                    model.state_dict(),
                    os.path.join(root_dir, "best_metric_model.pth"),
                )
                print("saved new best metric model")
            print(
                f"current epoch: {epoch + 1} current mean dice: {metric:.4f}"
                f" tc: {metric_tc:.4f} wt: {metric_wt:.4f} et: {metric_et:.4f}"
                f"\nbest mean dice: {best_metric:.4f}"
                f" at epoch: {best_metric_epoch}"
            )
    print(f"time consuming of epoch {epoch + 1} is: {(time.time() - epoch_start):.4f}")
total_time = time.time() - total_start